# WiseTech Global — WTC Intelligence Model V2

## Hybrid architecture

**Technical data + fundamentals + ASX/global market context + announcement/news sentiment**

Outputs for **1-day / 5-day / 20-day horizons**:

- Probability price is higher
- Expected forward return
- Classification validation
- Regression validation
- Feature importance
- Latest hybrid forecast

### Anti-leakage rule
Fundamentals and news should be timestamped by their **public release date/time**, not by the fiscal period they describe. This notebook only forward-fills a fundamental after the release date.

> Research tool only. Not financial advice or a live-trading system.


In [ ]:
# Run once if required
# !pip install -r requirements.txt


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wtc_model import *

FUNDAMENTALS_CSV = "data/fundamentals.csv"
SENTIMENT_CSV = "data/news_sentiment.csv"


## 1. Build point-in-time hybrid dataset

The starter project works even if the optional CSV files are empty. Add official WiseTech results to `data/fundamentals.csv` and announcement/news scoring to `data/news_sentiment.csv` to activate those layers.


In [ ]:
df = build_dataset(
    fundamentals_csv=FUNDAMENTALS_CSV,
    sentiment_csv=SENTIMENT_CSV
)
features = feature_columns(df)

print("Rows:", len(df))
print("Features:", len(features))
print("Latest market date:", df.index.max().date())
display(df.tail())


## 2. Inspect feature layers

In [ ]:
layers = {
    "Technical": [c for c in features if c.startswith("wtc_") or c in ["day_of_week","month"]],
    "Market": [c for c in features if c.startswith(("asx200_","audusd_","nasdaq_","sp500_","xro_","tne_","tech_peer_","risk_on_"))],
    "Fundamental": [c for c in features if c.startswith("fund_")],
    "Sentiment": [c for c in features if c.startswith(("sent_","news_","major_","days_since_"))],
}
for name, cols in layers.items():
    print(name, len(cols))
    print(cols[:8], "..." if len(cols)>8 else "")


## 3. Train multi-horizon classification + expected-return models

In [ ]:
models, metrics, predictions = train_horizon_models(df, features, test_fraction=0.20)
metrics_df = pd.DataFrame(metrics).T
metrics_df.index.name = "Horizon (days)"
display(metrics_df)


## 4. Latest WTC Intelligence forecast

In [ ]:
forecast = latest_forecast(df, features, models)
forecast["probability_up"] = forecast["probability_up"].map(lambda x: f"{x:.1%}")
forecast["expected_return"] = forecast["expected_return"].map(lambda x: f"{x:.2%}")
display(forecast)


## 5. Feature importance by horizon

In [ ]:
for h in HORIZONS:
    imp = pd.Series(
        models[h]["classifier"].feature_importances_,
        index=features
    ).sort_values(ascending=False).head(20)
    display(imp.to_frame(f"{h}d importance"))
    plt.figure(figsize=(10,7))
    imp.sort_values().plot(kind="barh")
    plt.title(f"WTC Intelligence Model — {h}-day direction feature importance")
    plt.tight_layout()
    plt.show()


## 6. Probability calibration diagnostics

In [ ]:
for h in HORIZONS:
    p = predictions[h].copy()
    p["probability_bucket"] = pd.cut(
        p["prob_up"], bins=[0,.35,.45,.55,.65,1],
        labels=["Very bearish","Bearish","Neutral","Bullish","Very bullish"]
    )
    calibration = p.groupby("probability_bucket", observed=True).agg(
        observations=("target_up_%dd" % h, "size"),
        mean_model_probability=("prob_up","mean"),
        actual_up_rate=("target_up_%dd" % h, "mean"),
        mean_expected_return=("expected_return","mean"),
        mean_actual_return=("future_return_%dd" % h,"mean")
    )
    print(f"\n{h}-day calibration")
    display(calibration)


## 7. Walk-forward validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, mean_absolute_error

for h in HORIZONS:
    d = df.dropna(subset=features + [f"future_return_{h}d", f"target_up_{h}d"]).copy()
    cv = TimeSeriesSplit(n_splits=5)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(d),1):
        tr, va = d.iloc[tr_idx], d.iloc[va_idx]
        clf, reg = make_classifier(), make_regressor()
        clf.fit(tr[features], tr[f"target_up_{h}d"])
        reg.fit(tr[features], tr[f"future_return_{h}d"])
        pp = clf.predict_proba(va[features])[:,1]
        rr = reg.predict(va[features])
        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(va[f"target_up_{h}d"], pp),
            "return_mae": mean_absolute_error(va[f"future_return_{h}d"], rr),
        })
    print(f"{h}-day")
    display(pd.DataFrame(rows))


## 8. Save model bundle

In [ ]:
Path("models").mkdir(exist_ok=True)
bundle = {
    "models": models,
    "features": features,
    "horizons": HORIZONS,
    "ticker": WTC,
    "training_end": str(df.dropna(subset=features).index.max().date()),
    "version": "2.0-hybrid",
}
joblib.dump(bundle, "models/wtc_intelligence_v2.joblib")
print("Saved models/wtc_intelligence_v2.joblib")
